# StyleModel 평가 노트북 — 어댑터 비교용

어댑터 zip 하나를 올리면, **베이스를 자동 인식**해 로드하고 고정 테스트셋으로 말투를 출력한다.
7B·14B 등 각각 올려 돌리고 결과를 비교한다. (greedy 디코딩 = 결정적이라 모델 간 공정 비교)

**런타임:** GPU(T4도 가능, 추론만이라 가벼움).

In [ ]:
!pip -q install -U "transformers>=4.44" "peft>=0.12" "bitsandbytes>=0.43" accelerate
import torch; print('cuda', torch.cuda.is_available())

In [ ]:
# 어댑터 zip 업로드 (예: epochs3_..._qwen7b.zip) 후 자동 압축해제
from google.colab import files
import zipfile, glob, os
up = files.upload()
zname = list(up.keys())[0]
with zipfile.ZipFile(zname) as z:
    z.extractall('adapter')
# adapter_config.json 위치 자동 탐색
cfgs = glob.glob('adapter/**/adapter_config.json', recursive=True)
ADAPTER = os.path.dirname(cfgs[0])
import json
BASE = json.load(open(cfgs[0]))['base_model_name_or_path']
print('adapter:', ADAPTER)
print('base   :', BASE)

In [ ]:
# 베이스(4비트) + 어댑터 로드
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
                         bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True)
tok = AutoTokenizer.from_pretrained(BASE)
if tok.pad_token is None: tok.pad_token = tok.eos_token
base = AutoModelForCausalLM.from_pretrained(BASE, quantization_config=bnb,
                                            device_map='auto', torch_dtype=torch.bfloat16)
model = PeftModel.from_pretrained(base, ADAPTER)
model.eval()
print('loaded:', BASE)

In [ ]:
# 고정 테스트셋 — 일상 / 한자어유발 / 자기합리화(실제 페르소나)
import torch
SYSTEM = ('너는 1930~40년대 소설가 이광수(춘원)다. 입력으로 주어진 평이한 현대 한국어 문장을, '
          '이광수 특유의 근대 국어 문체로 다시 써라. 한자어·격식체 어미·예스러운 어휘를 살리고 '
          '뜻은 그대로 보존하라. 변환한 문장만 출력하라.')
tests = [
    '나는 아침에 일찍 일어나 밥을 먹고 천천히 길을 걸었다.',
    '봄이 되니 마당의 나무에 새 잎이 돋고 꽃이 피었다.',
    '진정한 문명은 물질의 발전만이 아니라 정신의 성숙에서 온다.',
    '교육은 한 민족의 미래를 결정하는 가장 중요한 토대이다.',
    '나는 그것이 민족을 위한 불가피한 선택이었다고 믿는다.',
    '비난을 받을지라도 나는 시대의 흐름을 따랐을 뿐이다.',
]
for t in tests:
    msgs = [{'role':'system','content':SYSTEM},{'role':'user','content':t}]
    enc = tok.apply_chat_template(msgs, add_generation_prompt=True,
                                  return_tensors='pt', return_dict=True).to(model.device)
    with torch.no_grad():
        out = model.generate(**enc, max_new_tokens=220, do_sample=False)  # greedy=결정적
    gen = tok.decode(out[0][enc['input_ids'].shape[1]:], skip_special_tokens=True)
    print('[입력]', t)
    print('[근대체]', gen.strip())
    print('-'*64)